# 缠论多时间周期CTA策略 — 回测示例

本notebook演示如何使用vnpy回测框架对`CzscMultiTimeframeStrategy`策略进行回测分析。

## 环境要求
- Python >= 3.10
- vnpy >= 4.4.0
- vnpy_ctastrategy
- czsc（含Rust扩展）

## 回测流程
1. 配置数据库和历史数据
2. 初始化回测引擎
3. 配置策略参数
4. 运行回测
5. 分析绩效指标

In [ ]:
import sys
import os
from datetime import datetime

# 将策略目录加入路径
strategy_dir = os.path.dirname(os.path.abspath("__file__"))
if strategy_dir not in sys.path:
    sys.path.insert(0, strategy_dir)

# vnpy回测相关
from vnpy_ctastrategy.backtesting import BacktestingEngine, OptimizationSetting
from vnpy.trader.constant import Interval, Exchange
from vnpy.trader.object import BarData

# 策略导入
from czsc_multi_timeframe_strategy import CzscMultiTimeframeStrategy

print("导入成功")

## 回测配置说明

### 数据说明
vnpy回测需要历史K线数据存储在本地数据库中。支持的数据库：
- SQLite（默认，无需额外配置）
- MySQL、PostgreSQL、TDengine、QuestDB等

### 数据获取方式
1. 使用vnpy Data Manager从数据服务商下载
2. 通过vnpy RQData、XTData等数据源接口获取
3. 手动导入CSV格式的历史数据

### 合约说明
以下示例使用沪深300股指期货(IF)为例，可根据实际交易品种修改。

In [ ]:
# 创建回测引擎
engine = BacktestingEngine()

# 配置回测参数
engine.set_parameters(
    vt_symbol="IF888.CFFEX",       # 合约代码（主连）
    interval=Interval.MINUTE,      # 基础K线周期（1分钟）
    start=datetime(2022, 1, 1),    # 回测开始日期
    end=datetime(2024, 1, 1),      # 回测结束日期
    rate=0.0003,                   # 手续费率（万3）
    slippage=0.2,                  # 滑点（点数）
    size=300,                      # 合约乘数（IF为300）
    pricetick=0.2,                 # 最小价格变动单位
    capital=1_000_000,             # 初始资金（100万）
)

print("回测引擎配置完成")
print(f"合约: IF888.CFFEX")
print(f"周期: {datetime(2022,1,1).date()} ~ {datetime(2024,1,1).date()}")
print(f"初始资金: 1,000,000 元")

In [ ]:
# 策略参数配置
strategy_settings = {
    "fixed_size": 1,           # 每次交易1手
    "max_pos": 3,              # 最大持仓3手
    "stop_loss_pct": 0.02,     # 止损2%
    "take_profit_pct": 0.06,   # 止盈6%
    "trailing_stop_pct": 0.03, # 移动止损3%
    "min_bars_5m": 100,        # 5分钟最少100根K线
    "min_bars_30m": 50,        # 30分钟最少50根K线
    "min_bars_4h": 30,         # 4小时最少30根K线
    "support_tolerance_pct": 0.015,  # 支撑/阻力容忍度1.5%
}

# 添加策略到回测引擎
engine.add_strategy(
    CzscMultiTimeframeStrategy,
    strategy_settings
)

print("策略配置完成:")
for k, v in strategy_settings.items():
    print(f"  {k}: {v}")

In [ ]:
# 加载历史数据
print("正在加载历史数据...")
engine.load_data()

# 运行回测
print("正在运行回测...")
engine.run_backtesting()

print("回测完成！")

In [ ]:
# 计算绩效指标
df = engine.calculate_result()
stats = engine.calculate_statistics()

# 打印关键指标
print("=" * 50)
print("回测绩效指标")
print("=" * 50)
key_metrics = [
    "start_date", "end_date",
    "total_days", "profit_days", "loss_days",
    "capital", "end_balance",
    "total_return", "annual_return",
    "max_drawdown", "max_ddpercent",
    "total_net_pnl", "total_commission", "total_slippage",
    "total_trade_count",
    "daily_return", "return_std",
    "sharpe_ratio", "return_drawdown_ratio",
]

for metric in key_metrics:
    if metric in stats:
        print(f"{metric}: {stats[metric]}")

In [ ]:
# 绘制回测资金曲线
engine.show_chart()

## 参数优化

使用vnpy的OptimizationSetting进行参数扫描，寻找最优参数组合。

**注意**: 参数优化计算量大，建议先在小数据集上测试。

In [ ]:
# 参数优化配置
setting = OptimizationSetting()
setting.set_target("sharpe_ratio")   # 优化目标：夏普比率

# 参数搜索范围
setting.add_parameter("stop_loss_pct", 0.01, 0.04, 0.01)       # 1%~4%, 步长1%
setting.add_parameter("take_profit_pct", 0.04, 0.10, 0.02)     # 4%~10%, 步长2%
setting.add_parameter("trailing_stop_pct", 0.02, 0.05, 0.01)   # 2%~5%, 步长1%

print("参数优化范围:")
print(f"  stop_loss_pct: 1%~4% (步长1%)")
print(f"  take_profit_pct: 4%~10% (步长2%)")
print(f"  trailing_stop_pct: 2%~5% (步长1%)")
print(f"\n预计参数组合数量: {4 * 4 * 4} = 64 组")
print("\n运行优化（可能需要数分钟）...")

# 运行多进程优化（注释掉以避免意外执行）
# results = engine.run_optimization(setting)
# for result in results[:10]:  # 显示前10个最优结果
#     print(result)

print("\n如需运行优化，请取消注释上方代码")

## 策略优化建议

### 1. 信号增强
当前策略使用简化的背驰检测（笔幅度对比）。可以引入czsc的更多信号函数：

```python
# 使用czsc信号函数库
from czsc._native.signals import cxt_bi_status_V230101
```

### 2. 品种适配建议

| 品种类型 | 止损建议 | 止盈建议 | 备注 |
|---------|---------|---------|------|
| 股指期货 | 1.5%~2% | 5%~8% | 波动适中 |
| 商品期货 | 2%~3% | 6%~10% | 波动较大 |
| 股票 | 3%~5% | 10%~15% | 手续费低 |

### 3. 多品种扩展
策略可以同时运行在多个品种上，建议：
- 相关性低的品种组合（如股指+黄金+原油）
- 每个品种独立的策略实例
- 总仓位控制

### 4. 实盘注意事项
1. 先在模拟盘运行至少1个月
2. 关注滑点对收益的影响
3. 定期检查各周期数据的完整性
4. 注意czsc初始化期间的冷启动问题